# Notebook 5 — Spark Batch Processing & Feature Engineering
**Project**: Real-Time Retail Analytics & Demand Prediction Platform  
**Author**: Vineet Joshi | ZDA25M007 | IIT Madras Zanzibar  
**Stack**: Apache Spark 3.5 (bitnami cluster) | Delta Lake | MinIO  

Demonstrates distributed Spark processing on Delta Lake data: aggregations, window functions, and advanced feature engineering.

> Spark Master UI: **http://localhost:8083**

---

## 5.1 Spark Session

In [6]:
spark.stop()
print('Old Spark session stopped.')

Old Spark session stopped.


In [7]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

PACKAGES = (
    'io.delta:delta-spark_2.12:3.2.0,'
    'org.apache.hadoop:hadoop-aws:3.3.4,'
    'com.amazonaws:aws-java-sdk-bundle:1.12.367'
)

spark = (
    SparkSession.builder
    .appName('RetailBatchProcessing')
    .master('local[*]')
    .config('spark.jars.packages', PACKAGES)
    .config('spark.eventLog.enabled', 'false')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', 'admin')
    .config('spark.hadoop.fs.s3a.secret.key', 'bigdata123')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.sql.shuffle.partitions', '6')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready.')
print(f'Master: {spark.sparkContext.master}')

Spark 3.5.0 ready.
Master: local[*]


## 5.2 Load Scaled Dataset into Spark

In [10]:
import pyarrow.parquet as pq

# Load via pandas (avoids parquet timestamp issue)
PARQUET_PATH = '/home/jovyan/work/retail_20m.parquet'
pf = pq.ParquetFile(PARQUET_PATH)

# Read first row group only (~800K rows) to stay within memory
pdf = pf.read_row_group(0).to_pandas()
pdf = pdf[pdf['Quantity'] > 0]
pdf = pdf[pdf['UnitPrice'] > 0]
pdf['Revenue'] = pdf['Quantity'] * pdf['UnitPrice']

# Convert timestamps to string before passing to Spark
pdf['InvoiceDate'] = pdf['InvoiceDate'].astype(str)

# Create Spark DataFrame
df = spark.createDataFrame(pdf)
df = df.withColumn('InvoiceDate', F.to_timestamp('InvoiceDate'))
df.cache()

count = df.count()
print(f'Loaded {count:,} records into Spark.')
df.printSchema()

del pdf

Loaded 805,549 records into Spark.
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: long (nullable = true)
 |-- Country: string (nullable = true)
 |-- Revenue: double (nullable = true)



## 5.3 Distributed Aggregations

In [11]:
# Add time features
df = (
    df
    .withColumn('Hour',       F.hour('InvoiceDate'))
    .withColumn('DayOfWeek',  F.dayofweek('InvoiceDate'))
    .withColumn('Month',      F.month('InvoiceDate'))
    .withColumn('Year',       F.year('InvoiceDate'))
    .withColumn('WeekOfYear', F.weekofyear('InvoiceDate'))
    .withColumn('Date',       F.to_date('InvoiceDate'))
)

# Country-level aggregation
country_agg = (
    df.groupBy('Country')
    .agg(
        F.sum('Revenue').alias('TotalRevenue'),
        F.count('*').alias('Transactions'),
        F.countDistinct('CustomerID').alias('UniqueCustomers'),
        F.avg('UnitPrice').alias('AvgPrice')
    )
    .orderBy(F.desc('TotalRevenue'))
)

print('Top 10 Countries by Revenue:')
country_agg.show(10, truncate=False)

Top 10 Countries by Revenue:
+--------------+--------------------+------------+---------------+------------------+
|Country       |TotalRevenue        |Transactions|UniqueCustomers|AvgPrice          |
+--------------+--------------------+------------+---------------+------------------+
|United Kingdom|1.4729656136504129E7|725250      |5350           |3.0562150442791522|
|EIRE          |621974.878147115    |15743       |5              |5.317476058890211 |
|Netherlands   |553724.8421578609   |5088        |22             |2.684273510369468 |
|Germany       |431404.25970690895  |16694       |107            |3.594522795265965 |
|France        |355279.95910350233  |13812       |95             |4.281830402188983 |
|Australia     |169639.80682794296  |1812        |15             |3.571742057144727 |
|Spain         |109442.74709067194  |3719        |41             |4.250655519922488 |
|Switzerland   |100323.75954187685  |3011        |22             |3.8430558680091824|
|Sweden        |91418.239

In [12]:
# Monthly revenue trend
monthly = (
    df.groupBy('Year', 'Month')
    .agg(
        F.sum('Revenue').alias('MonthlyRevenue'),
        F.sum('Quantity').alias('MonthlyQty'),
        F.countDistinct('InvoiceNo').alias('MonthlyInvoices')
    )
    .orderBy('Year', 'Month')
)

print('Monthly Revenue Trend:')
monthly.show(20, truncate=False)

Monthly Revenue Trend:
+----+-----+------------------+----------+---------------+
|Year|Month|MonthlyRevenue    |MonthlyQty|MonthlyInvoices|
+----+-----+------------------+----------+---------------+
|2009|12   |686465.7481798392 |400153    |1512           |
|2010|1    |548184.4452304353 |365774    |1009           |
|2010|2    |506015.18570794817|373288    |1148           |
|2010|3    |697524.7525618935 |501865    |1575           |
|2010|4    |595581.9061066642 |351351    |1390           |
|2010|5    |610932.360132633  |393192    |1413           |
|2010|6    |621579.3309631713 |380931    |1493           |
|2010|7    |609326.2233045886 |336398    |1450           |
|2010|8    |590780.5674907306 |446946    |1288           |
|2010|9    |824107.9204246682 |562476    |1735           |
|2010|10   |1043095.6654130459|602758    |2202           |
|2010|11   |1161563.0983023457|652036    |2657           |
|2010|12   |910410.3452701293 |483451    |1510           |
|2011|1    |563253.285069647  |34

## 5.4 Window Functions — Product Ranking

In [13]:
# Rank products by revenue within each country
product_rev = (
    df.groupBy('Country', 'StockCode', 'Description')
    .agg(F.sum('Revenue').alias('ProductRevenue'))
)

w_rank = Window.partitionBy('Country').orderBy(F.desc('ProductRevenue'))
ranked = (
    product_rev
    .withColumn('Rank', F.rank().over(w_rank))
    .filter(F.col('Rank') <= 5)
)

print('Top 5 Products per Country (first 20 rows):')
ranked.show(20, truncate=30)

Top 5 Products per Country (first 20 rows):
+---------+---------+------------------------------+------------------+----+
|  Country|StockCode|                   Description|    ProductRevenue|Rank|
+---------+---------+------------------------------+------------------+----+
|Australia|    23084|            RABBIT NIGHT LIGHT|3336.1528598022383|   1|
|Australia|    22423|      REGENCY CAKESTAND 3 TIER|2881.3619585884944|   2|
|Australia|    21731| RED TOADSTOOL LED NIGHT LIGHT|2475.1015737807857|   3|
|Australia|    22630|          DOLLY GIRL LUNCH BOX|2191.6900826439473|   4|
|Australia|    22722|SET OF 6 SPICE TINS PANTRY ...|2086.3792977872918|   5|
|  Austria|     POST|                       POSTAGE|3057.6408382253016|   1|
|  Austria|   15056N|     EDWARDIAN PARASOL NATURAL| 670.6146514332129|   2|
|  Austria|  15056BL|       EDWARDIAN PARASOL BLACK| 586.0093176727438|   3|
|  Austria|    20679|         EDWARDIAN PARASOL RED| 584.3693134606222|   4|
|  Austria|   15056P|        EDW

## 5.5 Window Functions — Rolling Averages

In [14]:
# Daily revenue with 7-day rolling average
daily_rev = (
    df.groupBy('Date')
    .agg(F.sum('Revenue').alias('DailyRevenue'))
    .orderBy('Date')
)

w_rolling = Window.orderBy('Date').rowsBetween(-6, 0)
daily_with_ma = (
    daily_rev
    .withColumn('MA_7day', F.avg('DailyRevenue').over(w_rolling))
    .withColumn('CumulativeRevenue', F.sum('DailyRevenue').over(Window.orderBy('Date')))
)

print('Daily Revenue with 7-day Moving Average:')
daily_with_ma.show(15, truncate=False)

Daily Revenue with 7-day Moving Average:
+----------+------------------+------------------+------------------+
|Date      |DailyRevenue      |MA_7day           |CumulativeRevenue |
+----------+------------------+------------------+------------------+
|2009-12-01|21059.27585446205 |21059.27585446205 |21059.27585446205 |
|2009-12-02|46779.4716905492  |33919.37377250563 |67838.74754501125 |
|2009-12-03|62764.51044847295 |43534.4193311614  |130603.2579934842 |
|2009-12-04|50976.04840910377 |45394.82660064699 |181579.30640258797|
|2009-12-05|20671.032434703175|40450.067767458226|202250.33883729114|
|2009-12-06|17624.722529152972|36645.84356107402 |219875.0613664441 |
|2009-12-07|28950.790106224205|35546.55021038119 |248825.85147266832|
|2009-12-08|38923.963631121595|38098.64846418969 |287749.8151037899 |
|2009-12-09|37136.435005548294|36721.07179490385 |324886.2501093382 |
|2009-12-10|34739.86677685933 |32717.551270387623|359626.11688619753|
|2009-12-11|40970.360610969045|31288.167299225515

## 5.6 Advanced Feature Engineering for ML

In [15]:
# Product-level demand features with lag and rolling stats
product_daily = (
    df.groupBy('StockCode', 'Country', 'Date', 'Year', 'Month', 'WeekOfYear', 'DayOfWeek')
    .agg(
        F.sum('Quantity').alias('DailyQty'),
        F.sum('Revenue').alias('DailyRevenue'),
        F.avg('UnitPrice').alias('AvgPrice'),
        F.countDistinct('InvoiceNo').alias('NumInvoices'),
        F.countDistinct('CustomerID').alias('UniqueCustomers'),
        F.first('Description').alias('Description')
    )
    .filter(F.col('DailyQty') > 0)
)

# Add lag features
w_lag = Window.partitionBy('StockCode', 'Country').orderBy('Date')
product_features = (
    product_daily
    .withColumn('PrevDayQty',  F.lag('DailyQty', 1).over(w_lag))
    .withColumn('PrevDayRev',  F.lag('DailyRevenue', 1).over(w_lag))
    .withColumn('QtyChange',   F.col('DailyQty') - F.col('PrevDayQty'))
    .withColumn('Avg7DayQty',  F.avg('DailyQty').over(w_lag.rowsBetween(-6, 0)))
    .fillna(0, subset=['PrevDayQty', 'PrevDayRev', 'QtyChange', 'Avg7DayQty'])
)

feat_count = product_features.count()
print(f'Advanced feature table: {feat_count:,} rows')
product_features.printSchema()
product_features.show(5, truncate=20)

Advanced feature table: 506,906 rows
root
 |-- StockCode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- WeekOfYear: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DailyQty: long (nullable = true)
 |-- DailyRevenue: double (nullable = true)
 |-- AvgPrice: double (nullable = true)
 |-- NumInvoices: long (nullable = false)
 |-- UniqueCustomers: long (nullable = false)
 |-- Description: string (nullable = true)
 |-- PrevDayQty: long (nullable = true)
 |-- PrevDayRev: double (nullable = false)
 |-- QtyChange: long (nullable = true)
 |-- Avg7DayQty: double (nullable = false)

+---------+-------+----------+----+-----+----------+---------+--------+------------------+------------------+-----------+---------------+--------------------+----------+------------------+---------+----------+
|StockCode|Country|      Date|Year|Month|WeekOfYear|Day

## 5.7 Write Spark Results to Delta Lake

In [16]:
BUCKET = 'retail-v2'

# Write advanced features
product_features.write \
    .format('delta') \
    .mode('overwrite') \
    .save(f's3a://{BUCKET}/delta/spark_features')
print(f'✅ Spark features → s3a://{BUCKET}/delta/spark_features')

# Write country aggregation
country_agg.write \
    .format('delta') \
    .mode('overwrite') \
    .save(f's3a://{BUCKET}/delta/spark_country_agg')
print(f'✅ Country agg → s3a://{BUCKET}/delta/spark_country_agg')

# Write daily with MA
daily_with_ma.write \
    .format('delta') \
    .mode('overwrite') \
    .save(f's3a://{BUCKET}/delta/spark_daily_ma')
print(f'✅ Daily MA → s3a://{BUCKET}/delta/spark_daily_ma')

# Write monthly
monthly.write \
    .format('delta') \
    .mode('overwrite') \
    .save(f's3a://{BUCKET}/delta/spark_monthly')
print(f'✅ Monthly → s3a://{BUCKET}/delta/spark_monthly')

✅ Spark features → s3a://retail-v2/delta/spark_features
✅ Country agg → s3a://retail-v2/delta/spark_country_agg
✅ Daily MA → s3a://retail-v2/delta/spark_daily_ma
✅ Monthly → s3a://retail-v2/delta/spark_monthly


## 5.8 Spark Job Stats

In [17]:
print('='*55)
print('SPARK PROCESSING STATS')
print('='*55)
print(f'  Input records       : {count:,}')
print(f'  Feature rows        : {feat_count:,}')
print(f'  Countries           : {country_agg.count()}')
print(f'  Spark partitions    : {df.rdd.getNumPartitions()}')
print(f'  Executors           : 2 workers × 2 cores × 2GB')
print(f'  Shuffle partitions  : {spark.conf.get("spark.sql.shuffle.partitions")}')
print(f'\nSpark UI: http://localhost:8083')
print(f'MinIO:    http://localhost:9001 → retail-v2 bucket')

SPARK PROCESSING STATS
  Input records       : 805,549
  Feature rows        : 506,906
  Countries           : 41
  Spark partitions    : 4
  Executors           : 2 workers × 2 cores × 2GB
  Shuffle partitions  : 6

Spark UI: http://localhost:8083
MinIO:    http://localhost:9001 → retail-v2 bucket


## 5.9 Summary

In [18]:
print('='*55)
print('NOTEBOOK 5 COMPLETE')
print('='*55)
print(f'  Records processed   : {count:,} (distributed via Spark)')
print(f'  Feature rows        : {feat_count:,}')
print(f'  Aggregations        : country, monthly, daily MA, product ranking')
print(f'  Window functions    : rank, lag, rolling 7-day avg')
print()
print('Delta tables written to MinIO (retail-v2):')
print(f'  s3a://retail-v2/delta/spark_features')
print(f'  s3a://retail-v2/delta/spark_country_agg')
print(f'  s3a://retail-v2/delta/spark_daily_ma')
print(f'  s3a://retail-v2/delta/spark_monthly')
print()
print('Next: Notebook 6 — ML Training')

NOTEBOOK 5 COMPLETE
  Records processed   : 805,549 (distributed via Spark)
  Feature rows        : 506,906
  Aggregations        : country, monthly, daily MA, product ranking
  Window functions    : rank, lag, rolling 7-day avg

Delta tables written to MinIO (retail-v2):
  s3a://retail-v2/delta/spark_features
  s3a://retail-v2/delta/spark_country_agg
  s3a://retail-v2/delta/spark_daily_ma
  s3a://retail-v2/delta/spark_monthly

Next: Notebook 6 — ML Training
